In [ ]:
import torch
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, CrossEntropyLoss, Softplus, PReLU
from torch.utils.data import DataLoader
from torch.optim import Adam


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
dataset = load_wine()
dataset.data.shape
print(dataset.target_names)
print(dataset.feature_names)


In [ ]:
x = dataset.data
y = dataset.target

X_train,X_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)


In [ ]:
input_shape = len(dataset.feature_names)
output_shape = len(dataset.target_names)

model = Sequential(
    Linear(input_shape,4),
    ReLU(),
    Linear(4,8),
    ReLU(),
    Linear(8,output_shape)
)
model.to("cuda")
print(model)


In [ ]:
optmizer = Adam(model.parameters(), lr= 0.001)
criterion = CrossEntropyLoss().to("cuda")
max_epochs = 1000
batch = 8


In [ ]:
total_batchs = len(X_train) // batch
last_batch = len(X_train) % batch
has_last_b = last_batch > 0

print(total_batchs)
print(last_batch)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
).to(device)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
).to(device)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
).to(device)


In [ ]:
def train_model():

    model.train()

    for i_epochs in range(max_epochs):

        for i_batchs in range(total_batchs):

            inicio = i_batchs * batch
            fim = (i_batchs + 1) * batch

            batch_x = X_train_tensor[inicio:fim]
            batch_y = y_train_tensor[inicio:fim]

            optmizer.zero_grad()

            output = model(batch_x)
            loss = criterion(output, batch_y)

            loss.backward()
            optmizer.step()


In [ ]:
def eval_model():

    model.eval()

    with torch.no_grad():

        output = model(X_test_tensor)

        y_pred = torch.argmax(output, dim=1)

    y_pred = y_pred.cpu().numpy()

    print(f"Acuraçy Score: {(accuracy_score(y_test, y_pred)*100):.2f}%")
    print(f"F1 Score:  {(f1_score(y_test, y_pred,average="macro")*100):.2f}%")
    print(f"Confusion Matrix: \n{confusion_matrix(y_test,y_pred)}")


In [ ]:
train_model()
eval_model()


